In [2]:
!pip install requests pandas
!pip install wikipedia
!pip install beautifulsoup4
!pip install requests

In [3]:
import requests
import pandas as pd
import time

# 🔑 Your Watchmode API key
WATCHMODE_API_KEY = '5JWZ7PIkZd99c8eNHEzfpmxekHybpkpkJtfzOLNx'

# Base URL
base_url = 'https://api.watchmode.com/v1/list-titles/'


In [ ]:
# 🔑 Your TMDb API key
TMDB_API_KEY = '5cc24d059431e564a094997b6231486c'  # Replace with your actual TMDb key


In [ ]:
all_titles = []
limit = 10
max_pages = 2  # 40 × 250 = 10,000

for page in range(1, max_pages + 1):
    print(f"Fetching page {page}...")
    params = {
        'apiKey': WATCHMODE_API_KEY,
        'sources': 'amazon_prime',
        'regions': 'IN',
        'types': 'movie',
        'limit': limit,
        'page': page
    }
    response = requests.get(base_url, params=params)
    data = response.json()

    if 'titles' not in data or not data['titles']:
        print("No more titles returned. Stopping.")
        break

    all_titles.extend(data['titles'])
    time.sleep(0.5)

# Create DataFrame
df = pd.json_normalize(all_titles)
print(f"Total movies fetched: {len(df)}")
df.head()


Fetching page 1...
Fetching page 2...
Total movies fetched: 20


,id,title,year,imdb_id,tmdb_id,tmdb_type,type
0,1893263,KPop Demon Hunters,2025,tt14205554,803796,movie,movie
1,1577443,Countdown,2019,tt10039344,599975,movie,movie
2,1996084,Trainwreck: Mayor of Mayhem,2025,tt36856306,1484255,movie,movie
3,1942735,Semi-Soeter,2025,tt10727696,1428264,movie,movie
4,1538533,Ballerina,2025,tt7181546,541671,movie,movie


In [ ]:
# Fetch detailed movie data from Watchmode
def get_movie_details(movie_id):
    url = f'https://api.watchmode.com/v1/title/{movie_id}/details/'
    params = {'apiKey': WATCHMODE_API_KEY}
    r = requests.get(url, params=params)
    return r.json() if r.status_code == 200 else {}

# Fetch cast from TMDb using tmdb_id
def get_cast_from_tmdb(tmdb_id):
    url = f'https://api.themoviedb.org/3/movie/{tmdb_id}/credits'
    params = {'api_key': TMDB_API_KEY}
    r = requests.get(url, params=params)
    if r.status_code == 200:
        credits = r.json()
        cast_list = [member['name'] for member in credits.get('cast', [])[:5]]
        return ', '.join(cast_list)
    return 'Not Available'


In [ ]:
detailed_movies = []

# Use a smaller set at first if needed (e.g. df.head(200))
for idx, row in df.iterrows():
    movie_id = row['id']
    # tmdb_id = row.get('tmdb_id')

    # # Get details from Watchmode
    # details = get_movie_details(movie_id)
    # if not details:
    #     continue

    details = get_movie_details(movie_id)
    if not details:
        continue

# Use tmdb_id from details (more reliable than row)
    tmdb_id = details.get('tmdb_id')

# Get cast from TMDb
# cast = get_cast_from_tmdb(tmdb_id) if tmdb_id else 'Not Available'
    # Get cast from TMDb
    cast = get_cast_from_tmdb(tmdb_id) if tmdb_id else 'Not Available'

    detailed_movies.append({
        'Title': details.get('title'),
        'Year': details.get('year'),
        'Release Date': details.get('release_date'),
        'Genres': ', '.join(details.get('genre_names', [])),
        'Rating': details.get('user_rating'),
        'Cast': cast,
        'Plot Overview': details.get('plot_overview'),
        'Runtime (mins)': details.get('runtime_minutes'),
        'IMDb ID': details.get('imdb_id')
    })

    # Log progress
    if (idx + 1) % 50 == 0:
        print(f"Fetched {idx + 1} movies...")

    time.sleep(0.5)  # Avoid hitting rate limits


SSLError: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/803796/credits?api_key=69bf84f422ac92d2d001e76e2f0ef94f (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)')))

In [ ]:
final_df = pd.DataFrame(detailed_movies)

# Drop entries with missing required fields
final_df = final_df.dropna(subset=['Title', 'Release Date', 'Rating'])

# Convert release date to datetime and sort
final_df['Release Date'] = pd.to_datetime(final_df['Release Date'], errors='coerce')
final_df = final_df.sort_values(by='Release Date', ascending=False)

# Save to CSV
final_df.to_csv("amazon_prime_detailed_movies.csv", index=False)

# Preview
final_df.head(10)


,Title,Year,Release Date,Genres,Rating,Cast,Plot Overview,Runtime (mins),IMDb ID
16,Trainwreck: Poop Cruise,2025,2025-06-23,Documentary,5.9,,"An engine fire leaves 4,000 passengers strande...",55.0,tt36856455
0,KPop Demon Hunters,2025,2025-06-20,"Animation, Fantasy, Action, Comedy, Music",8.0,"Arden Cho, May Hong, Ji-young Yoo, Ahn Hyo-seo...","When K-pop superstars Rumi, Mira and Zoey aren...",96.0,tt14205554
3,Semi-Soeter,2025,2025-06-19,"Romance, Comedy",4.3,"Anel Alexander, Nico Panagio, Sandra Vaughn, L...",Power couple Jaci and JP find themselves in a ...,NaN,tt10727696
2,Trainwreck: Mayor of Mayhem,2025,2025-06-16,Documentary,6.4,"Rob Ford, Katie Simpson, Dave Rider, Tom Beyer...",Rob Ford scandalized Canadian politics as the ...,50.0,tt36856306
7,Deep Cover,2025,2025-06-12,"Action, Comedy, Crime",6.9,"Bryce Dallas Howard, Orlando Bloom, Nick Moham...",Kat is an improv comedy teacher beginning to q...,99.0,tt31121295
5,STRAW,2025,2025-06-05,"Thriller, Drama, Crime",6.7,"Taraji P. Henson, Sherri Shepherd, Teyana Tayl...",What will be her last straw? A devastatingly b...,105.0,tt32550101
4,Ballerina,2025,2025-06-04,"Action, Thriller, Crime",7.4,"Ana de Armas, Keanu Reeves, Ian McShane, Anjel...",Taking place during the events of John Wick: C...,125.0,tt7181546
18,Final Destination Bloodlines,2025,2025-05-14,"Horror, Mystery",7.1,"Kaitlyn Santa Juana, Teo Briones, Rya Kihlsted...","Plagued by a violent recurring nightmare, coll...",110.0,tt9619824
8,Sinners,2025,2025-04-16,"Horror, Thriller, Fantasy",7.9,"Michael B. Jordan, Hailee Steinfeld, Miles Cat...","Trying to leave their troubled lives behind, t...",138.0,tt31193180
6,Warfare,2025,2025-04-09,"War, Action",7.5,"D'Pharaoh Woon-A-Tai, Will Poulter, Cosmo Jarv...",A platoon of Navy SEALs embarks on a dangerous...,95.0,tt31434639


In [ ]:
from datetime import datetime

# Save file with today's date
today = datetime.today().strftime('%Y-%m-%d')
filename = f"prime_movies_{today}.csv"
final_df.to_csv(filename, index=False)
print(f"✅ Saved today's file as: {filename}")


✅ Saved today's file as: prime_movies_2025-07-01.csv


In [ ]:
import os
import glob
from datetime import datetime

# 🗂️ Step 1: Get today's date and file name
today = datetime.today().strftime('%Y-%m-%d')
print("Today is:", today)
today_file = f"prime_movies_{today}.csv"

# 🧠 Step 2: Find all matching files (exclude today's)
all_files = glob.glob("prime_movies_*.csv")
print("All matching files:", all_files)

previous_files = [f for f in all_files if not f.endswith(f"{today}.csv")]

if previous_files:
    # Step 3: Sort by date extracted from file name
    def extract_date_from_filename(filename):
        try:
            date_part = filename.split('_')[-1].replace('.csv', '')
            return datetime.strptime(date_part, "%Y-%m-%d")
        except:
            return datetime.min

    previous_files.sort(key=extract_date_from_filename, reverse=True)
    previous_file = previous_files[0]

    print(f"✅ Using previous file for comparison: {previous_file}")

    # Step 4: Load previous file
    prev_df = pd.read_csv(previous_file)

    # Step 5: Detect new movies (based on IMDb ID)
    new_movies = final_df[~final_df['IMDb ID'].isin(prev_df['IMDb ID'])]
    print(f"🆕 New movies found: {len(new_movies)}")

    # Step 6: Detect updated movies (rating or cast changed)
    updated_movies = final_df.merge(prev_df, on='IMDb ID', suffixes=('', '_old'))
    updated_movies = updated_movies[
        (updated_movies['Rating'] != updated_movies['Rating_old']) |
        (updated_movies['Cast'] != updated_movies['Cast_old'])
    ]
    print(f"🔁 Updated movies: {len(updated_movies)}")

    # Optional: Save results
    new_movies.to_csv(f"new_movies_{today}.csv", index=False)
    updated_movies.to_csv(f"updated_movies_{today}.csv", index=False)

else:
    print("⚠️ No previous file found. Skipping comparison.")


Today is: 2025-07-01
All matching files: ['prime_movies_2025-07-01.csv']
⚠️ No previous file found. Skipping comparison.


In [ ]:
!pip install wikipedia beautifulsoup4 requests
import wikipedia
from bs4 import BeautifulSoup
import requests


In [ ]:
# Wikipedia link
def get_wikipedia_url(title, year=None):
    try:
        search_query = f"{title} ({year}) film" if year else f"{title} film"
        page_title = wikipedia.search(search_query)[0]
        page = wikipedia.page(page_title, auto_suggest=False)
        return page.url
    except:
        return None


# Extract data from Wikipedia page
from bs4 import BeautifulSoup
import requests

def extract_wiki_data(url):
    try:
        # Fetch and parse the page
        res = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(res.text, 'html.parser')

        # ✅ Box Office
        box_office = None
        infobox = soup.find('table', class_='infobox')
        if infobox:
            for row in infobox.find_all('tr'):
                if row.th and 'box office' in row.th.text.lower():
                    box_office = row.td.get_text(separator=" ", strip=True)
                    break

        # ✅ Critical Response
        review = None
        for h3 in soup.find_all('h3'):
            span = h3.find('span', class_='mw-headline')
            if span and 'critical response' in span.text.lower():
                paragraphs = []
                for sibling in h3.find_next_siblings():
                    if sibling.name and sibling.name.startswith('h'):
                        break  # Stop at next section
                    if sibling.name == 'p':
                        para_text = sibling.get_text(strip=True)
                        if para_text:
                            paragraphs.append(para_text)
                review = ' '.join(paragraphs)
                break

        return box_office, review if review else None

    except Exception as e:
        print(f"❌ Error parsing {url}: {e}")
        return None, None






# IMDb data using OMDb API
OMDB_API_KEY = '566e8018E'  # Replace this!
def get_imdb_data(imdb_id):
    try:
        url = f"http://www.omdbapi.com/?i={imdb_id}&apikey={OMDB_API_KEY}"
        r = requests.get(url)
        data = r.json()
        return data.get('imdbRating'), data.get('Plot')
    except:
        return None, None


In [ ]:
from bs4 import BeautifulSoup
import requests
from bs4.element import Tag
import re

def extract_wiki_data(url):
    try:
        # Fetch and parse the Wikipedia page
        res = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(res.text, 'html.parser')

        # ✅ 1. Extract Box Office
        box_office = None
        infobox = soup.find('table', class_='infobox')
        if infobox:
            for row in infobox.find_all('tr'):
                if row.th and 'box office' in row.th.text.lower():
                    box_office = row.td.get_text(separator=" ", strip=True)
                    break

        # ✅ 2. Extract Critical Response Section
        review = None
        content_div = soup.find('div', id='mw-content-text')
        if not content_div:
            return box_office, None

        critical_header = None
        for tag in content_div.find_all(['h2', 'h3', 'h4']):
            span = tag.find('span', class_='mw-headline')
            if span and 'critical response' in span.text.strip().lower():
                critical_header = tag
                break

        if critical_header:
            paragraphs = []
            for elem in critical_header.next_siblings:
                if isinstance(elem, Tag) and elem.name and elem.name.startswith('h'):
                    # Stop at the next header of same or higher level
                    if int(elem.name[1]) <= int(critical_header.name[1]):
                        break
                if isinstance(elem, Tag) and elem.name == 'p':
                    text = elem.get_text(separator=' ', strip=True)
                    if text:
                        paragraphs.append(text)

            raw_review = ' '.join(paragraphs)
            # Clean citation markers like [123]
            review = re.sub(r'\[\s*\d+\s*\]', '', raw_review).strip()

        return box_office, review if review else None

    except Exception as e:
        print(f"❌ Error parsing {url}: {e}")
        return None, None


In [ ]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag
import re

# Wikipedia URL finder
def get_wikipedia_url(title, year=None):
    import wikipedia
    try:
        search_query = f"{title} ({year}) film" if year else f"{title} film"
        page_title = wikipedia.search(search_query)[0]
        page = wikipedia.page(page_title, auto_suggest=False)
        return page.url
    except:
        return None

# Extract data from Wikipedia (Box Office + Critical Response)
def extract_wiki_data(url):
    try:
        res = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(res.text, 'html.parser')

        # ✅ 1. Extract Box Office
        box_office = None
        infobox = soup.find('table', class_='infobox')
        if infobox:
            for row in infobox.find_all('tr'):
                if row.th and 'box office' in row.th.text.lower():
                    box_office = row.td.get_text(separator=" ", strip=True)
                    break

        # ✅ 2. Extract Critical Response
        review = extract_section_text_robust(soup, "Critical response")
        MAX_WORDS = 50            # ← change the limit here
        if review:
            words = review.split()
            review = " ".join(words[:MAX_WORDS]) + (" …" if len(words) > MAX_WORDS else "")
        return box_office, review if review else None

    except Exception as e:
        print(f"❌ Error parsing {url}: {e}")
        return None, None

# Helper function reused from your robust code
def extract_section_text_robust(soup, header_text):
    try:
        content_div = soup.find('div', id='mw-content-text')
        if not content_div:
            return None

        header_tag = None
        for tag in content_div.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6']):
            if tag.get_text(strip=True).lower() == header_text.lower():
                header_tag = tag
                break

        if not header_tag:
            return None

        texts = []
        for elem in header_tag.next_elements:
            if elem == header_tag:
                continue
            if isinstance(elem, Tag) and elem.name and elem.name.startswith('h'):
                if int(elem.name[1]) <= int(header_tag.name[1]):
                    break
            if isinstance(elem, Tag) and elem.name in ['p', 'ul', 'ol', 'dl']:
                text = elem.get_text(separator=' ', strip=True)
                if text:
                    texts.append(text)

        full_text = "\n\n".join(texts) if texts else None

        if full_text:
            clean_text = re.sub(r'\[\s*\d+\s*\]', '', full_text)
            return clean_text.strip()
        else:
            return None

    except Exception as e:
        print(f"❌ Error: {e}")
        return None




In [ ]:
final_df['Wikipedia URL'] = None
final_df['Box Office'] = None
final_df['Critical Review'] = None


In [ ]:
for i, row in final_df.iterrows():
    # print(f"🔍 Enriching: {row['Title']} ({row['Year']})")

    # Wikipedia
    wiki_url = get_wikipedia_url(row['Title'], row['Year'])
    final_df.at[i, 'Wikipedia URL'] = wiki_url

    if wiki_url:
        box_office, review = extract_wiki_data(wiki_url)
        final_df.at[i, 'Box Office'] = box_office
        final_df.at[i, 'Critical Review'] = review

    # For testing: stop early
    # if i == 10:
    #     break


In [ ]:
final_df.to_csv("prime_movies_enriched.csv", index=False)

In [ ]:
import requests

OMDB_API_KEY = '44436b53'  # Replace this if it's expired or rate-limited

def get_imdb_data(imdb_id):
    try:
        url = f"http://www.omdbapi.com/?i={imdb_id}&apikey={OMDB_API_KEY}"
        # print(f"🔎 Fetching OMDb: {url}")
        r = requests.get(url)
        data = r.json()

        if data.get('Response') == 'False':
            print(f"❌ OMDb Error: {data.get('Error')}")
            return None, None

        rating = data.get('imdbRating')
        plot = data.get('Plot')
        # print(f"✅ IMDb Rating: {rating}")
        # print(f"✅ Plot: {plot}")
        return rating, plot
    except Exception as e:
        print(f"❌ Exception during OMDb fetch: {e}")
        return None, None


In [ ]:
final_df['IMDb Rating'] = None
final_df['IMDb Plot'] = None

for i, row in final_df.iterrows():
    imdb_id = row.get('IMDb ID')  # Make sure you have 'imdbID' column with correct IDs
    if imdb_id:
        rating, plot = get_imdb_data(imdb_id)
        final_df.at[i, 'IMDb Rating'] = rating
        final_df.at[i, 'IMDb Plot'] = plot

final_df.to_csv("prime_movies_enriched.csv", index=False)


✅ We can deploy Colab logic on PythonAnywhere

✅ We just need to convert our notebook to .py and remove Colab-specific lines

✅ PythonAnywhere’s daily task scheduler takes care of automation and updated new released movies.

In [ ]:

import requests
import pandas as pd
from datetime import date

# Load existing dataset (assuming CSV previously saved)
try:
    df_existing = pd.read_csv('AmazonPrimeMovies_updated.csv')
except FileNotFoundError:
    df_existing = pd.DataFrame(columns=["title", "year", "type", "Date Added"])

# Function to fetch changed titles from Watchmode API
def fetch_new_watchmode_titles(api_key, since_date):
    changes_url = "https://api.watchmode.com/v1/changes/"
    params = {"apiKey": api_key, "changed_since": since_date}
    r = requests.get(changes_url, params=params)
    if r.status_code != 200:
        print("Failed to fetch changes.")
        return pd.DataFrame()

    changes = r.json()
    ids = [c["object_id"] for c in changes]
    titles = []

    for obj_id in ids:
        detail_url = f"https://api.watchmode.com/v1/title/{obj_id}/details/"
        detail_params = {"apiKey": api_key}
        t = requests.get(detail_url, params=detail_params).json()
        if t.get("name") and t.get("type"):
            titles.append({
                "title": t["name"],
                "year": t.get("year"),
                "type": t.get("type"),
                "Date Added": date.today().isoformat()
            })
    return pd.DataFrame(titles)

# Fetch today's new titles
today = date.today().isoformat()
df_new = fetch_new_watchmode_titles(WATCHMODE_API_KEY, since_date=today)

# Merge and deduplicate
df_combined = pd.concat([df_existing, df_new], ignore_index=True)
df_combined.drop_duplicates(subset=["title", "Date Added"], inplace=True)

# Save updated dataset
df_combined.to_csv('AmazonPrimeMovies_updated.csv', index=False)

# Preview latest entries
df_combined.tail()


Failed to fetch changes.


,title,year,type,Date Added
